# 🍛 Smart Canteen AI — Multi-Class Food Recognition Training (Google Colab GPU)

This notebook runs **High-Speed GPU Training** for the Sri Lankan Smart Canteen Food Recognition Subsystem.

### ⚡ Quickstart Steps:
1. In the Colab top menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Run **Step 1** to install libraries and verify your GPU.
3. Run **Step 2** to upload your `dataset_colab.zip` (generated locally via `python manage.py export-colab`).
4. Run **Step 3** to train the YOLO11 model in ~5 to 8 minutes!
5. Run **Step 4** to evaluate the model and automatically download `best.pt` to your computer.

## 1️⃣ Step 1: Install Ultralytics & Check Free GPU

In [ ]:
!pip install -q ultralytics

import torch
print("=" * 50)
if torch.cuda.is_available():
    print(f"✅ High-Speed GPU Active: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ GPU not detected! Please enable GPU in Runtime > Change runtime type > T4 GPU")
print("=" * 50)

## 2️⃣ Step 2: Upload Dataset
Run this cell and select the `data/dataset_colab.zip` file from your local project folder.

In [ ]:
from google.colab import files
import os
import shutil

# Upload dataset_colab.zip
print("📁 Please select 'dataset_colab.zip' to upload:")
uploaded = files.upload()

# Extract dataset
zip_name = list(uploaded.keys())[0]
!mkdir -p /content/dataset
!unzip -q -o "$zip_name" -d /content/dataset

print("\n✅ Dataset extracted successfully to /content/dataset!")
!cat /content/dataset/data.yaml

## 3️⃣ Step 3: Train YOLO11 Multi-Class Model on GPU (Takes ~5-8 mins)

In [ ]:
from ultralytics import YOLO

# Load lightweight YOLO11n backbone
model = YOLO('yolo11n.pt')

# Train with GPU acceleration
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=30,           # 30 epochs for high research accuracy
    batch=16,            # High batch size accelerated by T4 GPU
    imgsz=640,           # Standard resolution
    device=0,            # Use GPU 0
    patience=8,          # Early stopping patience
    project='/content/runs',
    name='canteen_food_model'
)

print("\n🎉 Training Complete!")

## 4️⃣ Step 4: Evaluate & Download `best.pt` to Your Computer

In [ ]:
import glob
from google.colab import files
from IPython.display import Image, display

# Display Confusion Matrix
cm_paths = glob.glob('/content/runs/canteen_food_model/confusion_matrix.png')
if cm_paths:
    print("📊 Model Confusion Matrix:")
    display(Image(cm_paths[0]))

# Display Results Graph
results_paths = glob.glob('/content/runs/canteen_food_model/results.png')
if results_paths:
    print("📈 Training Curves (mAP & Loss):")
    display(Image(results_paths[0]))

# Download the best trained weights file
best_weights = '/content/runs/canteen_food_model/weights/best.pt'
print(f"\n⬇️ Downloading '{best_weights}' to your computer...")
files.download(best_weights)